In [0]:
-- Databricks notebook
-- Purpose: Create and load the Silver Delta table tb_seguro_mensal_susep
-- Sources:
--   - susep_bronze.ses_seguros
--   - susep_bronze.aux_de_para_ramo_susep
--   - susep_bronze.aux_de_para_grupo_empresa
-- Target:
--   - susep_silver.tb_seguro_mensal_susep
--
-- Processing strategy:
--   - Recreate the table structure with CREATE OR REPLACE TABLE.
--   - Load data with INSERT OVERWRITE from Bronze to Silver.
--   - Keep full traceability and add market classification fields for insurance analytics.
--
-- New market classification fields:
--   - ds_categoria_mercado_susep
--   - fl_base_analitica_seguros
--   - fl_ramo_dpvat
--   - fl_ramo_acumulacao
--   - fl_ramo_previdencia
--   - fl_ramo_saude
--   - ds_criterio_classificacao_mercado

-- ============================================================
-- 1. CONFIGURATION
-- ============================================================

CREATE SCHEMA IF NOT EXISTS susep_silver;

SELECT
  'susep_bronze.ses_seguros' AS source_main_table,
  'susep_bronze.aux_de_para_ramo_susep' AS source_branch_mapping_table,
  'susep_bronze.aux_de_para_grupo_empresa' AS source_company_group_mapping_table,
  'susep_silver.tb_seguro_mensal_susep' AS target_table,
  CURRENT_TIMESTAMP() AS execution_timestamp;


In [0]:
-- ============================================================
-- 2. CREATE SILVER DELTA TABLE STRUCTURE
-- ============================================================
-- This step recreates the table with the expected schema, column comments,
-- Delta properties and partitioning strategy.
--
-- Important:
-- CREATE OR REPLACE TABLE is intentionally used because this notebook is
-- deterministic and the following step uses INSERT OVERWRITE.

CREATE OR REPLACE TABLE susep_silver.tb_seguro_mensal_susep (
  nr_ano_mes_referencia INT COMMENT 'Competência de referência no formato AAAAMM.',
  nr_ano_referencia INT COMMENT 'Ano da competência de referência.',
  nr_mes_referencia INT COMMENT 'Mês da competência de referência.',
  dt_referencia DATE COMMENT 'Data de referência da competência, sempre no primeiro dia do mês.',
  cd_entidade_susep STRING COMMENT 'Código SUSEP da entidade/seguradora.',
  nm_entidade_susep STRING COMMENT 'Nome da entidade/seguradora conforme base de empresas.',
  cd_grupo_empresa_susep_original STRING COMMENT 'Código original do grupo econômico informado na base SES seguros.',
  cd_grupo_empresa_susep_mapeado STRING COMMENT 'Código de grupo econômico após regra de de/para/deduplicação por entidade.',
  nm_grupo_empresa_susep_mapeado STRING COMMENT 'Nome do grupo econômico mapeado para a entidade.',
  cd_grupo_ramo_susep STRING COMMENT 'Código do grupo de ramos/produtos SUSEP.',
  nm_grupo_ramo_susep STRING COMMENT 'Nome do grupo de ramos/produtos SUSEP.',
  cd_ramo_susep STRING COMMENT 'Código do ramo SUSEP normalizado em 4 dígitos.',
  nm_ramo_susep STRING COMMENT 'Nome do ramo SUSEP.',
  cd_ramo_tratado STRING COMMENT 'Código tratado/agrupado do ramo para análise executiva.',
  nm_ramo_tratado STRING COMMENT 'Nome tratado/agrupado do ramo para análise executiva.',
  ds_categoria_mercado_susep STRING COMMENT 'Categoria de mercado atribuída ao ramo SUSEP para separar seguros, acumulação/VGBL, previdência/EFPC, saúde e DPVAT.',
  fl_base_analitica_seguros BOOLEAN COMMENT 'Indica se o registro deve compor a base analítica principal de seguros.',
  fl_ramo_dpvat BOOLEAN COMMENT 'Indica se o registro pertence ao ramo DPVAT.',
  fl_ramo_acumulacao BOOLEAN COMMENT 'Indica se o registro pertence a VGBL, sobrevivência ou outro produto de acumulação.',
  fl_ramo_previdencia BOOLEAN COMMENT 'Indica se o registro pertence a previdência, EFPC ou microsseguro previdenciário.',
  fl_ramo_saude BOOLEAN COMMENT 'Indica se o registro pertence a saúde suplementar ou saúde resseguro.',
  ds_criterio_classificacao_mercado STRING COMMENT 'Descrição da regra utilizada para classificar o registro na categoria de mercado SUSEP.',
  vl_premio_direto DECIMAL(24,6) COMMENT 'Valor de prêmio direto.',
  vl_premio_de_seguros DECIMAL(24,6) COMMENT 'Valor de prêmio de seguros.',
  vl_premio_retido DECIMAL(24,6) COMMENT 'Valor de prêmio retido.',
  vl_premio_ganho DECIMAL(24,6) COMMENT 'Valor de prêmio ganho.',
  vl_sinistro_direto DECIMAL(24,6) COMMENT 'Valor de sinistro direto.',
  vl_sinistro_retido DECIMAL(24,6) COMMENT 'Valor de sinistro retido.',
  vl_despesa_comercializacao DECIMAL(24,6) COMMENT 'Valor de despesa de comercialização/comissão.',
  vl_premio_emitido DECIMAL(24,6) COMMENT 'Valor de prêmio emitido.',
  vl_premio_emitido_capitalizacao DECIMAL(24,6) COMMENT 'Valor de prêmio emitido associado a capitalização quando presente na base.',
  vl_despesa_resseguro DECIMAL(24,6) COMMENT 'Valor de despesa com resseguro.',
  vl_sinistro_ocorrido DECIMAL(24,6) COMMENT 'Valor de sinistro ocorrido.',
  vl_receita_resseguro DECIMAL(24,6) COMMENT 'Valor de receita de resseguro.',
  vl_sinistro_ocorrido_capitalizacao DECIMAL(24,6) COMMENT 'Valor de sinistros ocorridos associados a capitalização quando presente na base.',
  vl_recuperacao_sinistro_ocorrido_capitalizacao DECIMAL(24,6) COMMENT 'Valor de recuperação de sinistros ocorridos associados a capitalização quando presente na base.',
  vl_rvne DECIMAL(24,6) COMMENT 'Valor de RVNE informado na base.',
  vl_convenio_dpvat DECIMAL(24,6) COMMENT 'Valor relacionado ao convênio DPVAT.',
  vl_consorcio_fundos DECIMAL(24,6) COMMENT 'Valor relacionado a consórcios e fundos.',
  ds_chave_busca_empresa STRING COMMENT 'Texto de busca usado para auditoria do mapeamento de empresa/grupo.',
  nm_empresa_consolidada_original STRING COMMENT 'Nome consolidado originalmente identificado na consulta/mapeamento.',
  nm_grupo_concorrencia_n1 STRING COMMENT 'Grupo concorrencial de primeiro nível.',
  nm_grupo_concorrencia_n2 STRING COMMENT 'Grupo concorrencial de segundo nível.',
  ds_tipo_grupo STRING COMMENT 'Classificação textual do tipo de grupo/empresa.',
  ds_regra_mapeamento_empresa STRING COMMENT 'Regra aplicada para consolidar a entidade no grupo concorrencial.',
  ds_confianca_mapeamento_empresa STRING COMMENT 'Nível de confiança do mapeamento de empresa.',
  ds_observacao_mapeamento_empresa STRING COMMENT 'Observação adicional sobre o mapeamento da empresa.',
  fl_alterou_vs_consulta_atual BOOLEAN COMMENT 'Indica se o mapeamento alterou o resultado originalmente consultado.',
  ds_regra_deduplicacao_empresa STRING COMMENT 'Regra de deduplicação aplicada na escolha da melhor linha por entidade.',
  dt_inicio_vigencia_mapeamento_ramo DATE COMMENT 'Data inicial de vigência da regra de ramo.',
  dt_fim_vigencia_mapeamento_ramo DATE COMMENT 'Data final de vigência da regra de ramo.',
  fl_ativo_mapeamento_ramo BOOLEAN COMMENT 'Indica se o mapeamento de ramo está ativo.',
  nr_versao_mapeamento_ramo STRING COMMENT 'Versão da tabela de mapeamento de ramo.',
  ds_justificativa_tratamento_ramo STRING COMMENT 'Justificativa do tratamento/agrupamento do ramo.',
  ds_fonte_regra_mapeamento_ramo STRING COMMENT 'Fonte ou racional da regra de mapeamento de ramo.',
  ts_processamento_mapeamento_ramo TIMESTAMP COMMENT 'Timestamp de processamento do de/para de ramo.',
  cd_hash_registro_mapeamento_ramo STRING COMMENT 'Hash do registro de mapeamento de ramo.',
  dt_inicio_vigencia_mapeamento_empresa DATE COMMENT 'Data inicial de vigência da regra de empresa.',
  dt_fim_vigencia_mapeamento_empresa DATE COMMENT 'Data final de vigência da regra de empresa.',
  fl_ativo_mapeamento_empresa BOOLEAN COMMENT 'Indica se o mapeamento de empresa está ativo.',
  nr_versao_mapeamento_empresa STRING COMMENT 'Versão da tabela de mapeamento de empresa/grupo.',
  ds_fonte_regra_mapeamento_empresa STRING COMMENT 'Fonte ou racional da regra de mapeamento de empresa.',
  ts_processamento_mapeamento_empresa TIMESTAMP COMMENT 'Timestamp de processamento do de/para de empresa.',
  cd_hash_registro_mapeamento_empresa STRING COMMENT 'Hash do registro de mapeamento de empresa.',
  ts_ingestao_bronze TIMESTAMP COMMENT 'Timestamp de ingestão da linha na Bronze.',
  nm_arquivo_origem STRING COMMENT 'Nome do arquivo de origem.',
  ds_caminho_arquivo_origem STRING COMMENT 'Caminho completo do arquivo de origem.',
  ds_pasta_origem STRING COMMENT 'Pasta de origem do arquivo carregado.',
  id_lote_ingestao STRING COMMENT 'Identificador do lote de ingestão da Bronze.',
  nm_tabela_origem STRING COMMENT 'Nome da tabela/arquivo de origem na ingestão.',
  nr_ano_mes_referencia_origem INT COMMENT 'Competência técnica de referência da origem, quando preenchida.',
  cd_hash_linha_bronze STRING COMMENT 'Hash da linha original na Bronze.',
  ts_processamento_silver TIMESTAMP COMMENT 'Timestamp de processamento/carga da Silver.',
  cd_hash_registro_silver STRING COMMENT 'Hash do registro já padronizado na Silver.'
)
USING DELTA
PARTITIONED BY (nr_ano_mes_referencia)
COMMENT 'Tabela Silver mensal de seguros SUSEP enriquecida com de/para de ramos, grupos concorrenciais de empresas e classificação de mercado para recorte analítico de seguros.'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'quality' = 'silver',
  'dominio' = 'seguros_susep',
  'projeto' = 'lakehouse_susep',
  'granularidade' = 'competencia_entidade_ramo',
  'origem_principal' = 'susep_bronze.ses_seguros',
  'classificacao_mercado' = 'seguros_danos_pessoas_dpvat_acumulacao_previdencia_saude'
);


In [0]:
-- ============================================================
-- 3. SOURCE TABLE VALIDATION
-- ============================================================
-- Quick validation before loading Silver.

SELECT
  'susep_bronze.ses_seguros' AS source_table,
  COUNT(*) AS qt_linhas
FROM susep_bronze.ses_seguros

UNION ALL

SELECT
  'susep_bronze.aux_de_para_ramo_susep' AS source_table,
  COUNT(*) AS qt_linhas
FROM susep_bronze.aux_de_para_ramo_susep

UNION ALL

SELECT
  'susep_bronze.aux_de_para_grupo_empresa' AS source_table,
  COUNT(*) AS qt_linhas
FROM susep_bronze.aux_de_para_grupo_empresa;


In [0]:
-- ============================================================
-- 4. LOAD SILVER TABLE FROM BRONZE
-- ============================================================
-- This step performs the joins, datatype treatment, code normalization,
-- date derivation, market classification and Silver hash generation.
-- ============================================================
-- 4. LOAD SILVER TABLE FROM BRONZE
-- ============================================================
-- This step performs the joins, datatype treatment, code normalization,
-- date derivation, market classification and Silver hash generation.
  INSERT OVERWRITE TABLE susep_silver.tb_seguro_mensal_susep (
  nr_ano_mes_referencia,
  nr_ano_referencia,
  nr_mes_referencia,
  dt_referencia,
  cd_entidade_susep,
  nm_entidade_susep,
  cd_grupo_empresa_susep_original,
  cd_grupo_empresa_susep_mapeado,
  nm_grupo_empresa_susep_mapeado,
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_susep,
  nm_ramo_susep,
  cd_ramo_tratado,
  nm_ramo_tratado,
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude,
  ds_criterio_classificacao_mercado,
  vl_premio_direto,
  vl_premio_de_seguros,
  vl_premio_retido,
  vl_premio_ganho,
  vl_sinistro_direto,
  vl_sinistro_retido,
  vl_despesa_comercializacao,
  vl_premio_emitido,
  vl_premio_emitido_capitalizacao,
  vl_despesa_resseguro,
  vl_sinistro_ocorrido,
  vl_receita_resseguro,
  vl_sinistro_ocorrido_capitalizacao,
  vl_recuperacao_sinistro_ocorrido_capitalizacao,
  vl_rvne,
  vl_convenio_dpvat,
  vl_consorcio_fundos,
  ds_chave_busca_empresa,
  nm_empresa_consolidada_original,
  nm_grupo_concorrencia_n1,
  nm_grupo_concorrencia_n2,
  ds_tipo_grupo,
  ds_regra_mapeamento_empresa,
  ds_confianca_mapeamento_empresa,
  ds_observacao_mapeamento_empresa,
  fl_alterou_vs_consulta_atual,
  ds_regra_deduplicacao_empresa,
  dt_inicio_vigencia_mapeamento_ramo,
  dt_fim_vigencia_mapeamento_ramo,
  fl_ativo_mapeamento_ramo,
  nr_versao_mapeamento_ramo,
  ds_justificativa_tratamento_ramo,
  ds_fonte_regra_mapeamento_ramo,
  ts_processamento_mapeamento_ramo,
  cd_hash_registro_mapeamento_ramo,
  dt_inicio_vigencia_mapeamento_empresa,
  dt_fim_vigencia_mapeamento_empresa,
  fl_ativo_mapeamento_empresa,
  nr_versao_mapeamento_empresa,
  ds_fonte_regra_mapeamento_empresa,
  ts_processamento_mapeamento_empresa,
  cd_hash_registro_mapeamento_empresa,
  ts_ingestao_bronze,
  nm_arquivo_origem,
  ds_caminho_arquivo_origem,
  ds_pasta_origem,
  id_lote_ingestao,
  nm_tabela_origem,
  nr_ano_mes_referencia_origem,
  cd_hash_linha_bronze,
  ts_processamento_silver,
  cd_hash_registro_silver
)
  WITH s_norm AS (SELECT
      s.*,
    cast(s.damesano as int) as nr_ano_mes_referencia,
    cast(SUBSTR(CAST(s.damesano AS STRING),
          1,
          4
        ) AS INT
      ) AS nr_ano_referencia,
     cast(SUBSTR(CAST(s.damesano AS STRING),
          5,
          2
        ) AS INT
      ) AS nr_mes_referencia,
      TO_DATE(
        CONCAT(
          cast(SUBSTR(CAST(s.damesano AS STRING),
          1,
          4
        ) AS INT
      ), '-',
       cast(SUBSTR(CAST(s.damesano AS STRING),
          5,
          2
        ) AS INT
      ),
          '-01'
        )
      ) AS dt_referencia,
      cast(s.coenti AS STRING) AS cd_entidade_susep,
      CAST(s.cogrupo AS STRING) AS cd_grupo_empresa_susep_original,
      CAST(s.coramo AS STRING) AS cd_ramo_susep
    FROM
      susep_bronze.ses_seguros s
  ),
  dados_silver AS (SELECT
      s_norm.nr_ano_mes_referencia,
      s_norm.nr_ano_referencia,
      s_norm.nr_mes_referencia,
      s_norm.dt_referencia,
      s_norm.cd_entidade_susep,
      e.noenti AS nm_entidade_susep,
      s_norm.cd_grupo_empresa_susep_original,
      e.cogrupo AS cd_grupo_empresa_susep_mapeado,
      e.nogrupo AS nm_grupo_empresa_susep_mapeado,
      r.cod_grupo_susep AS cd_grupo_ramo_susep,
     r.nome_grupo_susep AS nm_grupo_ramo_susep,
      s_norm.cd_ramo_susep,
      r.nome_ramo_susep AS nm_ramo_susep,
     r.cod_ramo_tratado AS cd_ramo_tratado,
     r.nome_ramo_tratado AS nm_ramo_tratado,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_direto AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_direto AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_direto AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_direto AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.premio_direto AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_direto AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_direto,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_de_seguros AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_de_seguros AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_de_seguros AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_de_seguros AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.premio_de_seguros AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_de_seguros AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_de_seguros,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_retido AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_retido AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_retido AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_retido AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.premio_retido AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_retido AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_retido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_ganho AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_ganho AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_ganho AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_ganho AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.premio_ganho AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_ganho AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_ganho,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistro_direto AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.sinistro_direto AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistro_direto AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistro_direto AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.sinistro_direto AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistro_direto AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_direto,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistro_retido AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.sinistro_retido AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistro_retido AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistro_retido AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.sinistro_retido AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistro_retido AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_retido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.desp_com AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.desp_com AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.desp_com AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.desp_com AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.desp_com AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.desp_com AS STRING)) AS DECIMAL(24, 6))
      END AS vl_despesa_comercializacao,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_emitido2 AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.premio_emitido2 AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_emitido2 AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_emitido2 AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.premio_emitido2 AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_emitido2 AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_emitido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(
            CAST(s_norm.premio_emitido_cap AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.premio_emitido_cap AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.premio_emitido_cap AS STRING)) AS DECIMAL(24, 6))
      END AS vl_premio_emitido_capitalizacao,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.despesa_resseguros AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(
            CAST(s_norm.despesa_resseguros AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.despesa_resseguros AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.despesa_resseguros AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.despesa_resseguros AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.despesa_resseguros AS STRING)) AS DECIMAL(24, 6))
      END AS vl_despesa_resseguro,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistro_ocorrido AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_ocorrido,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.receita_resseguro AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.receita_resseguro AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.receita_resseguro AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.receita_resseguro AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.receita_resseguro AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.receita_resseguro AS STRING)) AS DECIMAL(24, 6))
      END AS vl_receita_resseguro,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(
            CAST(s_norm.sinistros_ocorridos_cap AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.sinistros_ocorridos_cap AS STRING)) AS DECIMAL(24, 6))
      END AS vl_sinistro_ocorrido_capitalizacao,
      CASE
        WHEN
          NULLIF(TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)), 'null') IS NULL
        THEN
          NULL
        WHEN
          TRIM(
            CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)
          ) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        ELSE
          TRY_CAST(
            TRIM(CAST(s_norm.recuperacao_sinistros_ocorridos_cap AS STRING)) AS DECIMAL(24, 6)
          )
      END AS vl_recuperacao_sinistro_ocorrido_capitalizacao,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.rvne AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.rvne AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(REPLACE(TRIM(CAST(s_norm.rvne AS STRING)), '.', ''), ',', '.') AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.rvne AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.rvne AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.rvne AS STRING)) AS DECIMAL(24, 6))
      END AS vl_rvne,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.conveniodpvat AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.conveniodpvat AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.conveniodpvat AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.conveniodpvat AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(REPLACE(TRIM(CAST(s_norm.conveniodpvat AS STRING)), ',', '.') AS DECIMAL(24, 6))
        ELSE TRY_CAST(TRIM(CAST(s_norm.conveniodpvat AS STRING)) AS DECIMAL(24, 6))
      END AS vl_convenio_dpvat,
      CASE
        WHEN NULLIF(TRIM(CAST(s_norm.consorciosefundos AS STRING)), 'null') IS NULL THEN NULL
        WHEN
          TRIM(CAST(s_norm.consorciosefundos AS STRING)) RLIKE '^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(
              REPLACE(TRIM(CAST(s_norm.consorciosefundos AS STRING)), '.', ''),
              ',',
              '.'
            ) AS DECIMAL(24, 6)
          )
        WHEN
          TRIM(CAST(s_norm.consorciosefundos AS STRING)) RLIKE '^-?[0-9]+,[0-9]+$'
        THEN
          TRY_CAST(
            REPLACE(TRIM(CAST(s_norm.consorciosefundos AS STRING)), ',', '.') AS DECIMAL(24, 6)
          )
        ELSE TRY_CAST(TRIM(CAST(s_norm.consorciosefundos AS STRING)) AS DECIMAL(24, 6))
      END AS vl_consorcio_fundos,
      e.search_string AS ds_chave_busca_empresa,
     e.nome_empresa_consolidada_query_original AS nm_empresa_consolidada_original,
      e.grupo_concorrencia_n1 AS nm_grupo_concorrencia_n1,
      e.grupo_concorrencia_n2 AS nm_grupo_concorrencia_n2,
      e.tipo_grupo AS ds_tipo_grupo,
      e.regra_aplicada AS ds_regra_mapeamento_empresa,
     e.confianca_mapeamento AS ds_confianca_mapeamento_empresa,
     e.observacao  AS ds_observacao_mapeamento_empresa,
      CASE
        WHEN
          LOWER(TRIM(CAST(e.alterou_vs_query_atual AS STRING))) IN ('true', '1', 'sim', 's', 'yes')
        THEN
          TRUE
        WHEN
          LOWER(TRIM(CAST(e.alterou_vs_query_atual AS STRING))) IN (
            'false', '0', 'não', 'nao', 'n', 'no'
          )
        THEN
          FALSE
        ELSE NULL
      END AS fl_alterou_vs_consulta_atual,
      NULLIF(TRIM(CAST(e.dedup_rule_applied AS STRING)), 'null') AS ds_regra_deduplicacao_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(r.vigencia_inicio AS STRING)), 'null') AS DATE
      ) AS dt_inicio_vigencia_mapeamento_ramo,
      TRY_CAST(
        NULLIF(TRIM(CAST(r.vigencia_fim AS STRING)), 'null') AS DATE
      ) AS dt_fim_vigencia_mapeamento_ramo,
      CASE
        WHEN LOWER(TRIM(CAST(r.flag_ativo AS STRING))) IN ('true', '1', 'sim', 's', 'yes') THEN TRUE
        WHEN
          LOWER(TRIM(CAST(r.flag_ativo AS STRING))) IN ('false', '0', 'não', 'nao', 'n', 'no')
        THEN
          FALSE
        ELSE NULL
      END AS fl_ativo_mapeamento_ramo,
      NULLIF(TRIM(CAST(r.versao_mapeamento AS STRING)), 'null') AS nr_versao_mapeamento_ramo,
      NULLIF(
        TRIM(CAST(r.justificativa_tratamento AS STRING)),
        'null'
      ) AS ds_justificativa_tratamento_ramo,
      NULLIF(TRIM(CAST(r.fonte_regra AS STRING)), 'null') AS ds_fonte_regra_mapeamento_ramo,
      TRY_CAST(
        NULLIF(TRIM(CAST(r.dt_processamento AS STRING)), 'null') AS TIMESTAMP
      ) AS ts_processamento_mapeamento_ramo,
      NULLIF(TRIM(CAST(r.hash_registro AS STRING)), 'null') AS cd_hash_registro_mapeamento_ramo,
      TRY_CAST(
        NULLIF(TRIM(CAST(e.vigencia_inicio AS STRING)), 'null') AS DATE
      ) AS dt_inicio_vigencia_mapeamento_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(e.vigencia_fim AS STRING)), 'null') AS DATE
      ) AS dt_fim_vigencia_mapeamento_empresa,
      CASE
        WHEN LOWER(TRIM(CAST(e.flag_ativo AS STRING))) IN ('true', '1', 'sim', 's', 'yes') THEN TRUE
        WHEN
          LOWER(TRIM(CAST(e.flag_ativo AS STRING))) IN ('false', '0', 'não', 'nao', 'n', 'no')
        THEN
          FALSE
        ELSE NULL
      END AS fl_ativo_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.versao_mapeamento AS STRING)), 'null') AS nr_versao_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.fonte_regra AS STRING)), 'null') AS ds_fonte_regra_mapeamento_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(e.dt_processamento AS STRING)), 'null') AS TIMESTAMP
      ) AS ts_processamento_mapeamento_empresa,
      NULLIF(TRIM(CAST(e.hash_registro AS STRING)), 'null') AS cd_hash_registro_mapeamento_empresa,
      TRY_CAST(
        NULLIF(TRIM(CAST(s_norm.ingestion_timestamp AS STRING)), 'null') AS TIMESTAMP
      ) AS ts_ingestao_bronze,
      NULLIF(TRIM(CAST(s_norm.source_file AS STRING)), 'null') AS nm_arquivo_origem,
      NULLIF(TRIM(CAST(s_norm.source_file_path AS STRING)), 'null') AS ds_caminho_arquivo_origem,
      NULLIF(TRIM(CAST(s_norm.source_folder AS STRING)), 'null') AS ds_pasta_origem,
      NULLIF(TRIM(CAST(s_norm.ingestion_batch_id AS STRING)), 'null') AS id_lote_ingestao,
      NULLIF(TRIM(CAST(s_norm.source_table_name AS STRING)), 'null') AS nm_tabela_origem,
      TRY_CAST(
        REGEXP_REPLACE(
          NULLIF(TRIM(CAST(s_norm.reference_year_month AS STRING)), 'null'),
          '\\.0$',
          ''
        ) AS INT
      ) AS nr_ano_mes_referencia_origem,
      NULLIF(TRIM(CAST(s_norm.row_hash AS STRING)), 'null') AS cd_hash_linha_bronze
    FROM
      s_norm
        LEFT JOIN susep_bronze.aux_de_para_ramo_susep r
          ON s_norm.coramo
            = r.cod_ramo_susep 
          
        LEFT JOIN susep_bronze.aux_de_para_grupo_empresa e
          ON s_norm.coenti
            = e.coenti 
  ),
  base_classificacao_mercado AS (SELECT
      d.*,
      COALESCE(d.cd_ramo_tratado, d.cd_ramo_susep) AS cd_ramo_classificacao,
      COALESCE(
        d.cd_grupo_ramo_susep,
        SUBSTR(COALESCE(d.cd_ramo_tratado, d.cd_ramo_susep), 1, 2)
      ) AS cd_grupo_ramo_classificacao,
      UPPER(COALESCE(d.nm_ramo_tratado, d.nm_ramo_susep, '')) AS nm_ramo_classificacao
    FROM
      dados_silver d
  ),
  classificacao_silver AS (SELECT
      d.*,
      CASE
        WHEN d.cd_ramo_classificacao IS NULL THEN 'NAO_CLASSIFICADO'
        WHEN
          d.cd_ramo_classificacao IN ('0912', '1312')
          OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
        THEN
          'ACUMULACAO_VGBL_SOBREVIVENCIA'
        WHEN
          d.cd_ramo_classificacao = '1603'
          OR d.cd_grupo_ramo_classificacao = '22'
          OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
        THEN
          'PREVIDENCIA_EFPC'
        WHEN
          d.cd_grupo_ramo_classificacao = '19'
          OR d.cd_ramo_classificacao IN ('1202', '1901')
          OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
        THEN
          'SAUDE_SUPLEMENTAR'
        WHEN
          d.cd_ramo_classificacao = '0506'
          OR d.nm_ramo_classificacao RLIKE 'DPVAT'
        THEN
          'SEGURO_DPVAT'
        WHEN
          d.cd_grupo_ramo_classificacao IN ('09', '13')
          OR d.cd_ramo_classificacao RLIKE '^(09|13)'
        THEN
          'SEGUROS_PESSOAS_RISCO'
        WHEN
          d.cd_grupo_ramo_classificacao = '16'
          OR d.cd_ramo_classificacao RLIKE '^16'
        THEN
          'MICROSSEGUROS'
        ELSE 'SEGUROS_DANOS_RESPONSABILIDADES'
      END AS ds_categoria_mercado_susep,
      CASE
        WHEN
          (
            CASE
              WHEN d.cd_ramo_classificacao IS NULL THEN 'NAO_CLASSIFICADO'
              WHEN
                d.cd_ramo_classificacao IN ('0912', '1312')
                OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
              THEN
                'ACUMULACAO_VGBL_SOBREVIVENCIA'
              WHEN
                d.cd_ramo_classificacao = '1603'
                OR d.cd_grupo_ramo_classificacao = '22'
                OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
              THEN
                'PREVIDENCIA_EFPC'
              WHEN
                d.cd_grupo_ramo_classificacao = '19'
                OR d.cd_ramo_classificacao IN ('1202', '1901')
                OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
              THEN
                'SAUDE_SUPLEMENTAR'
              WHEN
                d.cd_ramo_classificacao = '0506'
                OR d.nm_ramo_classificacao RLIKE 'DPVAT'
              THEN
                'SEGURO_DPVAT'
              WHEN
                d.cd_grupo_ramo_classificacao IN ('09', '13')
                OR d.cd_ramo_classificacao RLIKE '^(09|13)'
              THEN
                'SEGUROS_PESSOAS_RISCO'
              WHEN
                d.cd_grupo_ramo_classificacao = '16'
                OR d.cd_ramo_classificacao RLIKE '^16'
              THEN
                'MICROSSEGUROS'
              ELSE 'SEGUROS_DANOS_RESPONSABILIDADES'
            END
          ) IN (
            'SEGUROS_DANOS_RESPONSABILIDADES',
            'SEGUROS_PESSOAS_RISCO',
            'MICROSSEGUROS',
            'SEGURO_DPVAT'
          )
        THEN
          TRUE
        ELSE FALSE
      END AS fl_base_analitica_seguros,
      CASE
        WHEN
          d.cd_ramo_classificacao = '0506'
          OR d.nm_ramo_classificacao RLIKE 'DPVAT'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_dpvat,
      CASE
        WHEN
          d.cd_ramo_classificacao IN ('0912', '1312')
          OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_acumulacao,
      CASE
        WHEN
          d.cd_ramo_classificacao = '1603'
          OR d.cd_grupo_ramo_classificacao = '22'
          OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_previdencia,
      CASE
        WHEN
          d.cd_grupo_ramo_classificacao = '19'
          OR d.cd_ramo_classificacao IN ('1202', '1901')
          OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
        THEN
          TRUE
        ELSE FALSE
      END AS fl_ramo_saude,
      CASE
        WHEN
          d.cd_ramo_classificacao IS NULL
        THEN
          'Não classificado porque não houve código de ramo suficiente para aplicar a regra de mercado.'
        WHEN
          d.cd_ramo_classificacao IN ('0912', '1312')
          OR d.nm_ramo_classificacao RLIKE 'VGBL|VAGP|VRGP|VRSA|VRID|VDR|SOBREVIV'
        THEN
          'Excluído da visão principal de seguros por regra de VGBL/sobrevivência/acumulação.'
        WHEN
          d.cd_ramo_classificacao = '1603'
          OR d.cd_grupo_ramo_classificacao = '22'
          OR d.nm_ramo_classificacao RLIKE 'PREVID|EFPC'
        THEN
          'Excluído da visão principal de seguros por regra de previdência, EFPC ou microsseguro previdenciário.'
        WHEN
          d.cd_grupo_ramo_classificacao = '19'
          OR d.cd_ramo_classificacao IN ('1202', '1901')
          OR d.nm_ramo_classificacao RLIKE 'SAUDE|SAÚDE'
        THEN
          'Excluído da visão principal de seguros por regra de saúde suplementar ou saúde resseguro.'
        WHEN
          d.cd_ramo_classificacao = '0506'
          OR d.nm_ramo_classificacao RLIKE 'DPVAT'
        THEN
          'Mantido na base de seguros, mas identificado por flag própria para análises com ou sem DPVAT.'
        WHEN
          d.cd_grupo_ramo_classificacao IN ('09', '13')
          OR d.cd_ramo_classificacao RLIKE '^(09|13)'
        THEN
          'Mantido na base de seguros como seguro de pessoas de risco, exceto regras específicas de acumulação já tratadas.'
        WHEN
          d.cd_grupo_ramo_classificacao = '16'
          OR d.cd_ramo_classificacao RLIKE '^16'
        THEN
          'Mantido na base de seguros como microsseguro, exceto microsseguro previdenciário já tratado.'
        ELSE
          'Mantido na base de seguros como seguro de danos, responsabilidades ou demais ramos securitários.'
      END AS ds_criterio_classificacao_mercado
    FROM
      base_classificacao_mercado d
  )
  SELECT
    d.nr_ano_mes_referencia,
    d.nr_ano_referencia,
    d.nr_mes_referencia,
    d.dt_referencia,
    d.cd_entidade_susep,
    d.nm_entidade_susep,
    d.cd_grupo_empresa_susep_original,
    d.cd_grupo_empresa_susep_mapeado,
    d.nm_grupo_empresa_susep_mapeado,
    d.cd_grupo_ramo_susep,
    d.nm_grupo_ramo_susep,
    d.cd_ramo_susep,
    d.nm_ramo_susep,
    d.cd_ramo_tratado,
    d.nm_ramo_tratado,
    d.ds_categoria_mercado_susep,
    d.fl_base_analitica_seguros,
    d.fl_ramo_dpvat,
    d.fl_ramo_acumulacao,
    d.fl_ramo_previdencia,
    d.fl_ramo_saude,
    d.ds_criterio_classificacao_mercado,
    d.vl_premio_direto,
    d.vl_premio_de_seguros,
    d.vl_premio_retido,
    d.vl_premio_ganho,
    d.vl_sinistro_direto,
    d.vl_sinistro_retido,
    d.vl_despesa_comercializacao,
    d.vl_premio_emitido,
    d.vl_premio_emitido_capitalizacao,
    d.vl_despesa_resseguro,
    d.vl_sinistro_ocorrido,
    d.vl_receita_resseguro,
    d.vl_sinistro_ocorrido_capitalizacao,
    d.vl_recuperacao_sinistro_ocorrido_capitalizacao,
    d.vl_rvne,
    d.vl_convenio_dpvat,
    d.vl_consorcio_fundos,
    d.ds_chave_busca_empresa,
    d.nm_empresa_consolidada_original,
    d.nm_grupo_concorrencia_n1,
    d.nm_grupo_concorrencia_n2,
    d.ds_tipo_grupo,
    d.ds_regra_mapeamento_empresa,
    d.ds_confianca_mapeamento_empresa,
    d.ds_observacao_mapeamento_empresa,
    d.fl_alterou_vs_consulta_atual,
    d.ds_regra_deduplicacao_empresa,
    d.dt_inicio_vigencia_mapeamento_ramo,
    d.dt_fim_vigencia_mapeamento_ramo,
    d.fl_ativo_mapeamento_ramo,
    d.nr_versao_mapeamento_ramo,
    d.ds_justificativa_tratamento_ramo,
    d.ds_fonte_regra_mapeamento_ramo,
    d.ts_processamento_mapeamento_ramo,
    d.cd_hash_registro_mapeamento_ramo,
    d.dt_inicio_vigencia_mapeamento_empresa,
    d.dt_fim_vigencia_mapeamento_empresa,
    d.fl_ativo_mapeamento_empresa,
    d.nr_versao_mapeamento_empresa,
    d.ds_fonte_regra_mapeamento_empresa,
    d.ts_processamento_mapeamento_empresa,
    d.cd_hash_registro_mapeamento_empresa,
    d.ts_ingestao_bronze,
    d.nm_arquivo_origem,
    d.ds_caminho_arquivo_origem,
    d.ds_pasta_origem,
    d.id_lote_ingestao,
    d.nm_tabela_origem,
    d.nr_ano_mes_referencia_origem,
    d.cd_hash_linha_bronze,
    CURRENT_TIMESTAMP() AS ts_processamento_silver,
    sha2(
      concat_ws(
        '||',
        COALESCE(CAST(d.nr_ano_mes_referencia AS STRING), ''),
        COALESCE(CAST(d.nr_ano_referencia AS STRING), ''),
        COALESCE(CAST(d.nr_mes_referencia AS STRING), ''),
        COALESCE(CAST(d.dt_referencia AS STRING), ''),
        COALESCE(CAST(d.cd_entidade_susep AS STRING), ''),
        COALESCE(CAST(d.nm_entidade_susep AS STRING), ''),
        COALESCE(CAST(d.cd_grupo_empresa_susep_original AS STRING), ''),
        COALESCE(CAST(d.cd_grupo_empresa_susep_mapeado AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_empresa_susep_mapeado AS STRING), ''),
        COALESCE(CAST(d.cd_grupo_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.cd_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.nm_ramo_susep AS STRING), ''),
        COALESCE(CAST(d.cd_ramo_tratado AS STRING), ''),
        COALESCE(CAST(d.nm_ramo_tratado AS STRING), ''),
        COALESCE(CAST(d.ds_categoria_mercado_susep AS STRING), ''),
        COALESCE(CAST(d.fl_base_analitica_seguros AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_dpvat AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_acumulacao AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_previdencia AS STRING), ''),
        COALESCE(CAST(d.fl_ramo_saude AS STRING), ''),
        COALESCE(CAST(d.ds_criterio_classificacao_mercado AS STRING), ''),
        COALESCE(CAST(d.vl_premio_direto AS STRING), ''),
        COALESCE(CAST(d.vl_premio_de_seguros AS STRING), ''),
        COALESCE(CAST(d.vl_premio_retido AS STRING), ''),
        COALESCE(CAST(d.vl_premio_ganho AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_direto AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_retido AS STRING), ''),
        COALESCE(CAST(d.vl_despesa_comercializacao AS STRING), ''),
        COALESCE(CAST(d.vl_premio_emitido AS STRING), ''),
        COALESCE(CAST(d.vl_premio_emitido_capitalizacao AS STRING), ''),
        COALESCE(CAST(d.vl_despesa_resseguro AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_ocorrido AS STRING), ''),
        COALESCE(CAST(d.vl_receita_resseguro AS STRING), ''),
        COALESCE(CAST(d.vl_sinistro_ocorrido_capitalizacao AS STRING), ''),
        COALESCE(CAST(d.vl_recuperacao_sinistro_ocorrido_capitalizacao AS STRING), ''),
        COALESCE(CAST(d.vl_rvne AS STRING), ''),
        COALESCE(CAST(d.vl_convenio_dpvat AS STRING), ''),
        COALESCE(CAST(d.vl_consorcio_fundos AS STRING), ''),
        COALESCE(CAST(d.ds_chave_busca_empresa AS STRING), ''),
        COALESCE(CAST(d.nm_empresa_consolidada_original AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_concorrencia_n1 AS STRING), ''),
        COALESCE(CAST(d.nm_grupo_concorrencia_n2 AS STRING), ''),
        COALESCE(CAST(d.ds_tipo_grupo AS STRING), ''),
        COALESCE(CAST(d.ds_regra_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ds_confianca_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ds_observacao_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.fl_alterou_vs_consulta_atual AS STRING), ''),
        COALESCE(CAST(d.ds_regra_deduplicacao_empresa AS STRING), ''),
        COALESCE(CAST(d.dt_inicio_vigencia_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.dt_fim_vigencia_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.fl_ativo_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.nr_versao_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.ds_justificativa_tratamento_ramo AS STRING), ''),
        COALESCE(CAST(d.ds_fonte_regra_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.ts_processamento_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.cd_hash_registro_mapeamento_ramo AS STRING), ''),
        COALESCE(CAST(d.dt_inicio_vigencia_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.dt_fim_vigencia_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.fl_ativo_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.nr_versao_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ds_fonte_regra_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ts_processamento_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.cd_hash_registro_mapeamento_empresa AS STRING), ''),
        COALESCE(CAST(d.ts_ingestao_bronze AS STRING), ''),
        COALESCE(CAST(d.nm_arquivo_origem AS STRING), ''),
        COALESCE(CAST(d.ds_caminho_arquivo_origem AS STRING), ''),
        COALESCE(CAST(d.ds_pasta_origem AS STRING), ''),
        COALESCE(CAST(d.id_lote_ingestao AS STRING), ''),
        COALESCE(CAST(d.nm_tabela_origem AS STRING), ''),
        COALESCE(CAST(d.nr_ano_mes_referencia_origem AS STRING), ''),
        COALESCE(CAST(d.cd_hash_linha_bronze AS STRING), '')
      ),
      256
    ) AS cd_hash_registro_silver
  FROM
    classificacao_silver d;

In [0]:
-- ============================================================
-- 5. CREATE ANALYTICAL VIEWS
-- ============================================================
-- These views preserve the full Silver table and make consumption easier
-- for insurance-specific analysis.

CREATE OR REPLACE VIEW susep_silver.vw_fato_seguro_mensal_susep AS
SELECT *
FROM susep_silver.tb_seguro_mensal_susep
WHERE fl_base_analitica_seguros = TRUE;

CREATE OR REPLACE VIEW susep_silver.vw_fato_seguro_mensal_susep_sem_dpvat AS
SELECT *
FROM susep_silver.tb_seguro_mensal_susep
WHERE fl_base_analitica_seguros = TRUE
  AND fl_ramo_dpvat = FALSE;

CREATE OR REPLACE VIEW susep_silver.vw_fato_seguro_mensal_susep_exclusoes AS
SELECT *
FROM susep_silver.tb_seguro_mensal_susep
WHERE fl_base_analitica_seguros = FALSE;


In [0]:
-- ============================================================
-- 6. ADD UNITY CATALOG TAGS
-- ============================================================
-- Execute this step in Unity Catalog enabled workspaces.
-- The table and column comments were already defined in the CREATE TABLE step.

ALTER TABLE susep_silver.tb_seguro_mensal_susep SET TAGS (
  'camada' = 'silver',
  'dominio_dado' = 'seguros_susep',
  'granularidade' = 'competencia_entidade_ramo',
  'projeto' = 'lakehouse_susep',
  'classificacao' = 'publico'
);

ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nr_ano_mes_referencia SET TAGS ('dominio_dado' = 'temporal', 'tipo_coluna' = 'chave_analitica', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nr_ano_referencia SET TAGS ('dominio_dado' = 'temporal', 'tipo_coluna' = 'derivado', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nr_mes_referencia SET TAGS ('dominio_dado' = 'temporal', 'tipo_coluna' = 'derivado', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN dt_referencia SET TAGS ('dominio_dado' = 'temporal', 'tipo_coluna' = 'derivado', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_entidade_susep SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'chave', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_entidade_susep SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'atributo', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_grupo_empresa_susep_original SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'chave', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_grupo_empresa_susep_mapeado SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'chave', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_grupo_empresa_susep_mapeado SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'atributo', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_grupo_ramo_susep SET TAGS ('dominio_dado' = 'produto', 'tipo_coluna' = 'chave', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_grupo_ramo_susep SET TAGS ('dominio_dado' = 'produto', 'tipo_coluna' = 'atributo', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_ramo_susep SET TAGS ('dominio_dado' = 'produto', 'tipo_coluna' = 'chave', 'origem' = 'ses_seguros___aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_ramo_susep SET TAGS ('dominio_dado' = 'produto', 'tipo_coluna' = 'atributo', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_ramo_tratado SET TAGS ('dominio_dado' = 'produto', 'tipo_coluna' = 'chave_curada', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_ramo_tratado SET TAGS ('dominio_dado' = 'produto', 'tipo_coluna' = 'atributo_curado', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_categoria_mercado_susep SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'classificacao', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_base_analitica_seguros SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'flag_analitica', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_ramo_dpvat SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'flag_classificacao', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_ramo_acumulacao SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'flag_classificacao', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_ramo_previdencia SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'flag_classificacao', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_ramo_saude SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'flag_classificacao', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_criterio_classificacao_mercado SET TAGS ('dominio_dado' = 'mercado_susep', 'tipo_coluna' = 'auditoria_regra', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_premio_direto SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_premio_de_seguros SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_premio_retido SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_premio_ganho SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_sinistro_direto SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_sinistro_retido SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_despesa_comercializacao SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_premio_emitido SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_premio_emitido_capitalizacao SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_despesa_resseguro SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_sinistro_ocorrido SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_receita_resseguro SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_sinistro_ocorrido_capitalizacao SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_recuperacao_sinistro_ocorrido_capitalizacao SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_rvne SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_convenio_dpvat SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN vl_consorcio_fundos SET TAGS ('dominio_dado' = 'financeiro', 'tipo_coluna' = 'medida', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_chave_busca_empresa SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'auditoria', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_empresa_consolidada_original SET TAGS ('dominio_dado' = 'empresa', 'tipo_coluna' = 'atributo_curado', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_grupo_concorrencia_n1 SET TAGS ('dominio_dado' = 'concorrencia', 'tipo_coluna' = 'atributo_curado', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_grupo_concorrencia_n2 SET TAGS ('dominio_dado' = 'concorrencia', 'tipo_coluna' = 'atributo_curado', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_tipo_grupo SET TAGS ('dominio_dado' = 'concorrencia', 'tipo_coluna' = 'atributo_curado', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_regra_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'regra', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_confianca_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'qualidade', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_observacao_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'observacao', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_alterou_vs_consulta_atual SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'flag', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_regra_deduplicacao_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'regra', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN dt_inicio_vigencia_mapeamento_ramo SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'vigencia', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN dt_fim_vigencia_mapeamento_ramo SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'vigencia', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_ativo_mapeamento_ramo SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'flag', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nr_versao_mapeamento_ramo SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'versao', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_justificativa_tratamento_ramo SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'justificativa', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_fonte_regra_mapeamento_ramo SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'fonte', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ts_processamento_mapeamento_ramo SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'timestamp', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_hash_registro_mapeamento_ramo SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'hash', 'origem' = 'aux_de_para_ramo_susep');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN dt_inicio_vigencia_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'vigencia', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN dt_fim_vigencia_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'vigencia', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN fl_ativo_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'flag', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nr_versao_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'versao', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_fonte_regra_mapeamento_empresa SET TAGS ('dominio_dado' = 'governanca', 'tipo_coluna' = 'fonte', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ts_processamento_mapeamento_empresa SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'timestamp', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_hash_registro_mapeamento_empresa SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'hash', 'origem' = 'aux_de_para_grupo_empresa');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ts_ingestao_bronze SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'timestamp', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_arquivo_origem SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'arquivo', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_caminho_arquivo_origem SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'arquivo', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ds_pasta_origem SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'arquivo', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN id_lote_ingestao SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'lote', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nm_tabela_origem SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'tabela', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN nr_ano_mes_referencia_origem SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'referencia', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_hash_linha_bronze SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'hash', 'origem' = 'ses_seguros');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN ts_processamento_silver SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'timestamp', 'origem' = 'derivado');
ALTER TABLE susep_silver.tb_seguro_mensal_susep ALTER COLUMN cd_hash_registro_silver SET TAGS ('dominio_dado' = 'linhagem', 'tipo_coluna' = 'hash', 'origem' = 'derivado');


In [0]:
-- ============================================================
-- 7. POST LOAD CHECK - VOLUME AND BASIC COVERAGE
-- ============================================================

SELECT COUNT(*) AS qt_linhas_silver
FROM susep_silver.tb_seguro_mensal_susep;

SELECT
  COUNT(*) AS qt_linhas,
  COUNT_IF(nm_ramo_susep IS NULL) AS qt_sem_de_para_ramo,
  COUNT_IF(nm_entidade_susep IS NULL) AS qt_sem_de_para_empresa,
  COUNT_IF(ds_categoria_mercado_susep IS NULL) AS qt_sem_categoria_mercado,
  MIN(dt_referencia) AS dt_referencia_min,
  MAX(dt_referencia) AS dt_referencia_max
FROM susep_silver.tb_seguro_mensal_susep;


In [0]:
-- ============================================================
-- 8. POST LOAD CHECK - MARKET CLASSIFICATION
-- ============================================================

SELECT
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude,
  COUNT(*) AS qt_linhas,
  SUM(vl_premio_direto) AS vl_premio_direto,
  SUM(vl_premio_emitido_capitalizacao) AS vl_premio_emitido_capitalizacao
FROM susep_silver.tb_seguro_mensal_susep
GROUP BY
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude
ORDER BY qt_linhas DESC;


In [0]:
-- ============================================================
-- 9. POST LOAD CHECK - MONTHLY INSURANCE VIEW
-- ============================================================

SELECT
  nr_ano_mes_referencia,
  COUNT(*) AS qt_linhas,
  SUM(vl_premio_direto) AS vl_premio_direto_total,
  SUM(CASE WHEN fl_base_analitica_seguros THEN vl_premio_direto ELSE 0 END) AS vl_premio_direto_base_seguros,
  SUM(CASE WHEN fl_base_analitica_seguros AND NOT fl_ramo_dpvat THEN vl_premio_direto ELSE 0 END) AS vl_premio_direto_base_seguros_sem_dpvat,
  SUM(CASE WHEN fl_ramo_acumulacao THEN vl_premio_direto ELSE 0 END) AS vl_premio_direto_acumulacao,
  SUM(CASE WHEN fl_ramo_previdencia THEN vl_premio_direto ELSE 0 END) AS vl_premio_direto_previdencia,
  SUM(CASE WHEN fl_ramo_saude THEN vl_premio_direto ELSE 0 END) AS vl_premio_direto_saude
FROM susep_silver.tb_seguro_mensal_susep
GROUP BY nr_ano_mes_referencia
ORDER BY nr_ano_mes_referencia DESC
LIMIT 24;


In [0]:
-- ============================================================
-- 10. POST LOAD CHECK - EXCLUDED BRANCHES DETAIL
-- ============================================================
-- Use this query to audit what was excluded from the main insurance view.

SELECT
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_tratado,
  nm_ramo_tratado,
  ds_categoria_mercado_susep,
  ds_criterio_classificacao_mercado,
  COUNT(*) AS qt_linhas,
  SUM(vl_premio_direto) AS vl_premio_direto
FROM susep_silver.tb_seguro_mensal_susep
WHERE fl_base_analitica_seguros = FALSE
GROUP BY
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_tratado,
  nm_ramo_tratado,
  ds_categoria_mercado_susep,
  ds_criterio_classificacao_mercado
ORDER BY vl_premio_direto DESC;


In [0]:
-- ============================================================
-- 11. OPTIONAL OPTIMIZATION
-- ============================================================
-- For larger historical loads, this can improve query performance.
-- Keep this step optional for academic/MVP execution if cluster cost is a concern.

OPTIMIZE susep_silver.tb_seguro_mensal_susep
ZORDER BY (nr_ano_mes_referencia, cd_entidade_susep, cd_ramo_tratado);
